<a href="https://colab.research.google.com/github/sheliabond/Prescriptive-Analytics---Spring-2026-Public-/blob/main/Assignments/Final_Project/%20Final_Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Final Project**: The Organizational Decision Playbook

Student Name: Shelia Bond

Date: April 20, 2026

**Overview**

Your final project asks you to apply the full prescriptive analytics toolkit — everything from Lessons 1 through 10 — to a real decision problem in your own professional domain. You will build a working optimization model, test its assumptions, and present your findings to a fictional VP who will challenge you on your choices.



**Learning Objectives**

By completing this project, you will demonstrate that you can:

*   Distinguish a prescriptive problem from a descriptive or predictive one and articulate why optimization applies (Lesson 1)
*   Frame a business decision precisely: objectives, decision
variables, constraints, and tradeoffs (Lesson 2)
*  Build and solve a linear and/or integer optimization model using PuLP (Lessons 3, 4, 8)
*   Connect model outputs to real-world implementation considerations (Lesson 5)
*  Test model assumptions through sensitivity analysis and identify critical parameters (Lesson 6)
*   Incorporate a time dimension into your analysis (Lesson 9)
*  Identify where linear assumptions break down and explain the implications honestly (Lesson 10)














In [5]:
# Install required packages (if needed in Colab)
# Skip if running locally and packages are already installed
%pip install pulp pandas matplotlib numpy -q

In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pulp import LpMaximize, LpMinimize, LpProblem, LpVariable, lpSum, value, LpStatus, PULP_CBC_CMD
import io

print("Libraries imported successfully!")

Libraries imported successfully!



**Section 1** — Why This Problem Is Prescriptive

The indirect auto lending portfolio flows through a network of dealerships, but the credit union does not allocate lending capacity equally among them. Descriptive analytics explains how the portfolio has historically performed across dealers, while predictive analytics estimates future outcomes such as losses, yields, and volumes. However, neither approach determines how to optimally allocate lending capacity, as this requires balancing tradeoffs between risk, return, and growth objectives. Prescriptive analytics applies by identifying the optimal allocation strategy that maximizes performance within defined constraints.

**Decision Statement**: The Vice President of Consumer Lending must decide how to allocate the indirect auto lending budget across active dealerships. Success is defined by maximizing net yield while maintaining risk within approved limits and achieving loan growth targets.


**Section 2** — Decision Framing

Define each of the following clearly. This is not optional detail — a weak framing produces a weak model.

**Decision variables** — The model decides how
much lending volume to allocate to each dealer

**Objective** — Maximize total net yield across all allocated dealers.

**Constraints**


1.   Total allocated volume must not exceed \$25M per quarter
2.   Total allocation must be at least \$18M to meet loan growth targets

1.   Portfolio risk must stay within the approved loss limit ( risk score < 2.8)
2.   No single dealer can recieve more than \$3M per quarter



**Key tradeoffs**

Higher yield comes with higher risk, and pushing for growth can conflict with staying within risk limits. There is also a balance between concentrating volume in top dealers and maintaining diversification







**Section 3** — The Optimization Model

Build and solve a PuLP model that includes:

*   A linear objective function
*   At least two meaningful constraints that reflect real business rules
*   At least one integer or binary decision variable — a yes/no or whole-number choice


Solve the model and interpret the result in plain language: what does the model recommend, and what does that mean in practice? If your model is infeasible, diagnose which constraints conflict, relax one with justification, and report the tradeoff.

### Optimization Model Implementation

First, let's create some dummy data for our dealers. This data will include their potential yield rate and risk score, which are crucial for the optimization model.

In [11]:
import pandas as pd
import numpy as np

# Set a seed for reproducibility
np.random.seed(42)

dealer_names = [
    'BMW Dealer', 'Mercedes Dealer', 'Nissan Dealer', 'Toyota Dealer',
    'Mazda Dealer', 'Dodge Dealer', 'Acura Dealer', 'Ford Dealer',
    'Honda Dealer', 'Kia Dealer'
]

# Generate random data for Risk Score and Potential Yield Rate
# Risk Score between 1.5 and 3.5
risk_scores = np.round(np.random.uniform(1.5, 3.5, len(dealer_names)), 2).tolist() # Round to 2 decimal places
# Potential Yield Rate between 4% and 8%, rounded to 2 decimal places for percentage (i.e., 4 decimal places for the float)
yield_rates = np.round(np.random.uniform(0.04, 0.08, len(dealer_names)), 4).tolist()

dealer_data = {
    'Dealer_ID': dealer_names,
    'Potential_Yield_Rate': yield_rates,
    'Risk_Score': risk_scores
}
dealers_df = pd.DataFrame(dealer_data)

# Ensure Dealer_ID is the index for easier lookup
dealers_df = dealers_df.set_index('Dealer_ID')

print("New Dummy Dealer Data:")
# Display with formatted yield rate and risk score
display(dealers_df.head(10).style.format({
    'Potential_Yield_Rate': '{:.2%}',
    'Risk_Score': '{:.2f}' # Format Risk_Score to two decimal places
})) # Display all 10 dealers

New Dummy Dealer Data:


,Potential_Yield_Rate,Risk_Score
Dealer_ID,,
BMW Dealer,4.08%,2.25
Mercedes Dealer,7.88%,3.40
Nissan Dealer,7.33%,2.96
Toyota Dealer,4.85%,2.70
Mazda Dealer,4.73%,1.81
Dodge Dealer,4.73%,1.81
Acura Dealer,5.22%,1.62
Ford Dealer,6.10%,3.23
Honda Dealer,5.73%,2.70


In [12]:
from pulp import LpMaximize, LpMinimize, LpProblem, LpVariable, lpSum, value, LpStatus, PULP_CBC_CMD

# Create the problem instance, aiming to maximize
model = LpProblem("Indirect_Auto_Lending_Allocation", LpMaximize)

dealers = dealers_df.index.tolist()

# Decision Variables
# x[d] is the allocated volume to dealer d in millions USD
x = LpVariable.dicts("Allocation", dealers, lowBound=0, cat='Continuous')
# y[d] is 1 if dealer d is selected, 0 otherwise (binary variable)
y = LpVariable.dicts("Dealer_Selected", dealers, cat='Binary')

# Objective Function: Maximize total net yield
model += lpSum(x[d] * dealers_df.loc[d, 'Potential_Yield_Rate'] for d in dealers), "Total Net Yield"

# Constraints
# 1. Total allocated volume must not exceed $25M
model += lpSum(x[d] for d in dealers) <= 25, "Max Total Allocation"

# 2. Total allocation must be at least $18M (loan growth target)
model += lpSum(x[d] for d in dealers) >= 18, "Min Total Allocation_Growth Target"

# 3. Portfolio risk must stay within the approved loss limit (risk score < 2.8)
# Linearized form: sum(x[d] * risk_score[d]) < 2.8 * sum(x[d])
# We use 2.799 instead of 2.8 to ensure strict inequality and avoid numerical issues at the boundary
model += lpSum(x[d] * dealers_df.loc[d, 'Risk_Score'] for d in dealers) <= 2.799 * lpSum(x[d] for d in dealers), "Portfolio Risk Limit"

# 4. No single dealer can receive more than $3M per quarter
# Also links x[d] to y[d]: if y[d] is 0, x[d] must be 0
for d in dealers:
    model += x[d] <= 3 * y[d], f"Max Allocation for Dealer {d}"
    # An implicit constraint from the above: if y[d]=0, x[d] must be 0. If y[d]=1, x[d] <= 3.

# Solve the model
model.solve(PULP_CBC_CMD(msg=0)) # msg=0 to suppress solver output

print(f"Model Status: {LpStatus[model.status]}")

if model.status == 1: # Optimal
    print(f"Total Net Yield (Millions USD): {value(model.objective):.4f}")
    print("\nAllocation per Dealer (Millions USD):")
    results = []
    for d in dealers:
        if x[d].varValue > 0.001: # Only show dealers with significant allocation
            results.append({
                'Dealer_ID': d,
                'Allocated_Volume': x[d].varValue,
                'Selected': y[d].varValue
            })
    results_df = pd.DataFrame(results).set_index('Dealer_ID')
    display(results_df)

    # Calculate actual average risk score of allocated portfolio
    total_allocated_volume = lpSum(x[d].varValue for d in dealers).value()
    total_weighted_risk = lpSum(x[d].varValue * dealers_df.loc[d, 'Risk_Score'] for d in dealers).value()
    if total_allocated_volume > 0:
        actual_avg_risk = total_weighted_risk / total_allocated_volume
        print(f"\nTotal Allocated Volume (Millions USD): {total_allocated_volume:.2f}")
        print(f"Actual Average Portfolio Risk Score: {actual_avg_risk:.2f}")
    else:
        print("\nNo volume was allocated.")
elif model.status == -1: # Infeasible
    print("The model is infeasible. This means there is no solution that satisfies all constraints.")
    print("You may need to review and relax some constraints.")
else:
    print("The model could not be solved to optimality.")

Model Status: Optimal
Total Net Yield (Millions USD): 1.4573

Allocation per Dealer (Millions USD):


,Allocated_Volume,Selected
Dealer_ID,,
Mercedes Dealer,3.0,1.0
Nissan Dealer,3.0,1.0
Toyota Dealer,3.0,1.0
Mazda Dealer,3.0,1.0
Dodge Dealer,1.0,1.0
Acura Dealer,3.0,1.0
Ford Dealer,3.0,1.0
Honda Dealer,3.0,1.0
Kia Dealer,3.0,1.0



Total Allocated Volume (Millions USD): 25.00
Actual Average Portfolio Risk Score: 2.63


**Section 4** — Sensitivity Analysis

Identify the three parameters you are least confident about. For each:

*   Vary it by ±20% and show how the recommendation changes
*   State whether the recommendation is robust or fragile to that parameter

End with 2–3 sentences answering: how confident should the VP be in this recommendation, and under what conditions would it change?

In [15]:
import pandas as pd
import numpy as np
from pulp import LpMaximize, LpMinimize, LpProblem, LpVariable, lpSum, value, LpStatus, PULP_CBC_CMD

# Assuming dealers_df is already defined from previous cells
# If not, ensure it's available in the environment
if 'dealers_df' not in locals():
    print("dealers_df not found. Please run previous cells.")
    # Recreate dummy data for demonstration if not present
    np.random.seed(42)
    dealer_names = [
        'BMW Dealer', 'Mercedes Dealer', 'Nissan Dealer', 'Toyota Dealer',
        'Mazda Dealer', 'Dodge Dealer', 'Acura Dealer', 'Ford Dealer',
        'Honda Dealer', 'Kia Dealer'
    ]
    risk_scores = np.round(np.random.uniform(1.5, 3.5, len(dealer_names)), 2).tolist()
    yield_rates = np.round(np.random.uniform(0.04, 0.08, len(dealer_names)), 4).tolist()
    dealer_data = {
        'Dealer_ID': dealer_names,
        'Potential_Yield_Rate': yield_rates,
        'Risk_Score': risk_scores
    }
    dealers_df = pd.DataFrame(dealer_data).set_index('Dealer_ID')


def solve_lending_model(
    min_total_allocation: float,
    risk_limit: float,
    max_dealer_allocation: float
):
    """
    Builds and solves the indirect auto lending allocation optimization model
    with given parameters.

    Args:
        min_total_allocation (float): Minimum total allocation in millions USD.
        risk_limit (float): Maximum allowed average portfolio risk score.
        max_dealer_allocation (float): Maximum allocation for a single dealer in millions USD.

    Returns:
        dict: A dictionary containing model status, objective value, actual average risk,
              total allocated volume, and a DataFrame of dealer allocations.
    """
    model = LpProblem("Indirect_Auto_Lending_Allocation", LpMaximize)
    dealers = dealers_df.index.tolist()

    x = LpVariable.dicts("Allocation", dealers, lowBound=0, cat='Continuous')
    y = LpVariable.dicts("Dealer_Selected", dealers, cat='Binary')

    model += lpSum(x[d] * dealers_df.loc[d, 'Potential_Yield_Rate'] for d in dealers), "Total Net Yield"

    model += lpSum(x[d] for d in dealers) <= 25, "Max Total Allocation"
    model += lpSum(x[d] for d in dealers) >= min_total_allocation, "Min Total Allocation_Growth Target"
    model += lpSum(x[d] * dealers_df.loc[d, 'Risk_Score'] for d in dealers) <= (risk_limit - 0.001) * lpSum(x[d] for d in dealers), "Portfolio Risk Limit"

    for d in dealers:
        model += x[d] <= max_dealer_allocation * y[d], f"Max Allocation for Dealer {d}"

    model.solve(PULP_CBC_CMD(msg=0))

    results = {
        'status': LpStatus[model.status],
        'objective_value': None,
        'total_allocated_volume': None,
        'actual_avg_risk': None,
        'allocations': pd.DataFrame()
    }

    if model.status == 1:
        results['objective_value'] = value(model.objective)
        total_allocated_volume = lpSum(x[d].varValue for d in dealers).value()
        total_weighted_risk = lpSum(x[d].varValue * dealers_df.loc[d, 'Risk_Score'] for d in dealers).value()

        results['total_allocated_volume'] = total_allocated_volume
        if total_allocated_volume > 0:
            results['actual_avg_risk'] = total_weighted_risk / total_allocated_volume

        alloc_data = []
        for d in dealers:
            if x[d].varValue > 0.001:
                alloc_data.append({
                    'Dealer_ID': d,
                    'Allocated_Volume': x[d].varValue,
                    'Selected': y[d].varValue
                })
        results['allocations'] = pd.DataFrame(alloc_data).set_index('Dealer_ID')

    return results

# Original Parameters
ORIG_MIN_ALLOC = 18.0
ORIG_RISK_LIMIT = 2.8
ORIG_MAX_DEALER_ALLOC = 3.0

print("--- Original Model Results ---")
original_results = solve_lending_model(ORIG_MIN_ALLOC, ORIG_RISK_LIMIT, ORIG_MAX_DEALER_ALLOC)
if original_results['status'] == 'Optimal':
    print(f"Total Net Yield: {original_results['objective_value']:.4f}")
    print(f"Total Allocated Volume: {original_results['total_allocated_volume']:.2f}")
    print(f"Actual Average Risk Score: {original_results['actual_avg_risk']:.2f}")
    print("Allocations:")
    display(original_results['allocations'])
else:
    print(f"Original model status: {original_results['status']}")

# --- Sensitivity Analysis: Minimum Total Allocation (Loan Growth Target) ---
print("\n--- Sensitivity Analysis: Minimum Total Allocation ---")

# Vary by -20%
min_alloc_minus_20 = ORIG_MIN_ALLOC * 0.8
print(f"\nScenario 1: Minimum Total Allocation = ${min_alloc_minus_20:.2f}M (-20%)")
results_min_alloc_minus_20 = solve_lending_model(min_alloc_minus_20, ORIG_RISK_LIMIT, ORIG_MAX_DEALER_ALLOC)
if results_min_alloc_minus_20['status'] == 'Optimal':
    print(f"Total Net Yield: {results_min_alloc_minus_20['objective_value']:.4f}")
    print(f"Total Allocated Volume: {results_min_alloc_minus_20['total_allocated_volume']:.2f}")
    print(f"Actual Average Risk Score: {results_min_alloc_minus_20['actual_avg_risk']:.2f}")
    # display(results_min_alloc_minus_20['allocations'])
else:
    print(f"Model status: {results_min_alloc_minus_20['status']}")

# Vary by +20%
min_alloc_plus_20 = ORIG_MIN_ALLOC * 1.2
print(f"\nScenario 2: Minimum Total Allocation = ${min_alloc_plus_20:.2f}M (+20%)")
results_min_alloc_plus_20 = solve_lending_model(min_alloc_plus_20, ORIG_RISK_LIMIT, ORIG_MAX_DEALER_ALLOC)
if results_min_alloc_plus_20['status'] == 'Optimal':
    print(f"Total Net Yield: {results_min_alloc_plus_20['objective_value']:.4f}")
    print(f"Total Allocated Volume: {results_min_alloc_plus_20['total_allocated_volume']:.2f}")
    print(f"Actual Average Risk Score: {results_min_alloc_plus_20['actual_avg_risk']:.2f}")
    # display(results_min_alloc_plus_20['allocations'])
else:
    print(f"Model status: {results_min_alloc_plus_20['status']}")

# Comparison and Robustness for Min Total Allocation
print("\nComparison for Minimum Total Allocation:")
print(f"Original Net Yield: {original_results['objective_value']:.4f}")
print(f"-20% Min Alloc Net Yield: {results_min_alloc_minus_20['objective_value']:.4f}")
print(f"+20% Min Alloc Net Yield: {results_min_alloc_plus_20['objective_value']:.4f}")

if abs(original_results['objective_value'] - results_min_alloc_minus_20['objective_value']) / original_results['objective_value'] < 0.05 and \
   abs(original_results['objective_value'] - results_min_alloc_plus_20['objective_value']) / original_results['objective_value'] < 0.05:
    print("Recommendation for 'Minimum Total Allocation' seems relatively robust to small changes in this parameter, as the objective function (Total Net Yield) changes by less than 5% for a 20% variation.")
else:
    print("Recommendation for 'Minimum Total Allocation' seems somewhat fragile to changes in this parameter, as the objective function (Total Net Yield) changes significantly for a 20% variation. Further analysis of individual allocations might reveal more about fragility.")

# --- Sensitivity Analysis: Portfolio Risk Limit ---
print("\n--- Sensitivity Analysis: Portfolio Risk Limit ---")

# Vary by -20% (Stricter Risk Limit)
risk_limit_minus_20 = ORIG_RISK_LIMIT * 0.8
print(f"\nScenario 3: Portfolio Risk Limit = {risk_limit_minus_20:.2f} (-20%)")
results_risk_minus_20 = solve_lending_model(ORIG_MIN_ALLOC, risk_limit_minus_20, ORIG_MAX_DEALER_ALLOC)
if results_risk_minus_20['status'] == 'Optimal':
    print(f"Total Net Yield: {results_risk_minus_20['objective_value']:.4f}")
    print(f"Total Allocated Volume: {results_risk_minus_20['total_allocated_volume']:.2f}")
    print(f"Actual Average Risk Score: {results_risk_minus_20['actual_avg_risk']:.2f}")
    # display(results_risk_minus_20['allocations'])
else:
    print(f"Model status: {results_risk_minus_20['status']}")

# Vary by +20% (Looser Risk Limit)
risk_limit_plus_20 = ORIG_RISK_LIMIT * 1.2
print(f"\nScenario 4: Portfolio Risk Limit = {risk_limit_plus_20:.2f} (+20%)")
results_risk_plus_20 = solve_lending_model(ORIG_MIN_ALLOC, risk_limit_plus_20, ORIG_MAX_DEALER_ALLOC)
if results_risk_plus_20['status'] == 'Optimal':
    print(f"Total Net Yield: {results_risk_plus_20['objective_value']:.4f}")
    print(f"Total Allocated Volume: {results_risk_plus_20['total_allocated_volume']:.2f}")
    print(f"Actual Average Risk Score: {results_risk_plus_20['actual_avg_risk']:.2f}")
    # display(results_risk_plus_20['allocations'])
else:
    print(f"Model status: {results_risk_plus_20['status']}")

# Comparison and Robustness for Risk Limit
print("\nComparison for Portfolio Risk Limit:")
print(f"Original Net Yield: {original_results['objective_value']:.4f}")
print(f"-20% Risk Limit Net Yield: {results_risk_minus_20['objective_value']:.4f}")
print(f"+20% Risk Limit Net Yield: {results_risk_plus_20['objective_value']:.4f}")

if results_risk_minus_20['objective_value'] is not None and original_results['objective_value'] is not None and \
   abs(original_results['objective_value'] - results_risk_minus_20['objective_value']) / original_results['objective_value'] < 0.05 and \
   results_risk_plus_20['objective_value'] is not None and \
   abs(original_results['objective_value'] - results_risk_plus_20['objective_value']) / original_results['objective_value'] < 0.05:
    print("Recommendation for 'Portfolio Risk Limit' seems relatively robust to small changes in this parameter, as the objective function (Total Net Yield) changes by less than 5% for a 20% variation.")
else:
    print("Recommendation for 'Portfolio Risk Limit' seems somewhat fragile to changes in this parameter, as the objective function (Total Net Yield) changes significantly for a 20% variation.")

# --- Sensitivity Analysis: Max Dealer Allocation ---
print("\n--- Sensitivity Analysis: Max Dealer Allocation ---")

# Vary by -20% (Stricter max dealer allocation)
max_dealer_alloc_minus_20 = ORIG_MAX_DEALER_ALLOC * 0.8
print(f"\nScenario 5: Max Dealer Allocation = ${max_dealer_alloc_minus_20:.2f}M (-20%)")
results_max_dealer_alloc_minus_20 = solve_lending_model(ORIG_MIN_ALLOC, ORIG_RISK_LIMIT, max_dealer_alloc_minus_20)
if results_max_dealer_alloc_minus_20['status'] == 'Optimal':
    print(f"Total Net Yield: {results_max_dealer_alloc_minus_20['objective_value']:.4f}")
    print(f"Total Allocated Volume: {results_max_dealer_alloc_minus_20['total_allocated_volume']:.2f}")
    print(f"Actual Average Risk Score: {results_max_dealer_alloc_minus_20['actual_avg_risk']:.2f}")
    # display(results_max_dealer_alloc_minus_20['allocations'])
else:
    print(f"Model status: {results_max_dealer_alloc_minus_20['status']}")

# Vary by +20% (Looser max dealer allocation)
max_dealer_alloc_plus_20 = ORIG_MAX_DEALER_ALLOC * 1.2
print(f"\nScenario 6: Max Dealer Allocation = ${max_dealer_alloc_plus_20:.2f}M (+20%)")
results_max_dealer_alloc_plus_20 = solve_lending_model(ORIG_MIN_ALLOC, ORIG_RISK_LIMIT, max_dealer_alloc_plus_20)
if results_max_dealer_alloc_plus_20['status'] == 'Optimal':
    print(f"Total Net Yield: {results_max_dealer_alloc_plus_20['objective_value']:.4f}")
    print(f"Total Allocated Volume: {results_max_dealer_alloc_plus_20['total_allocated_volume']:.2f}")
    print(f"Actual Average Risk Score: {results_max_dealer_alloc_plus_20['actual_avg_risk']:.2f}")
    # display(results_max_dealer_alloc_plus_20['allocations'])
else:
    print(f"Model status: {results_max_dealer_alloc_plus_20['status']}")

# Comparison and Robustness for Max Dealer Allocation
print("\nComparison for Max Dealer Allocation:")
print(f"Original Net Yield: {original_results['objective_value']:.4f}")
print(f"-20% Max Dealer Alloc Net Yield: {results_max_dealer_alloc_minus_20['objective_value']:.4f}")
print(f"+20% Max Dealer Alloc Net Yield: {results_max_dealer_alloc_plus_20['objective_value']:.4f}")

if results_max_dealer_alloc_minus_20['objective_value'] is not None and original_results['objective_value'] is not None and \
   abs(original_results['objective_value'] - results_max_dealer_alloc_minus_20['objective_value']) / original_results['objective_value'] < 0.05 and \
   results_max_dealer_alloc_plus_20['objective_value'] is not None and \
   abs(original_results['objective_value'] - results_max_dealer_alloc_plus_20['objective_value']) / original_results['objective_value'] < 0.05:
    print("Recommendation for 'Max Dealer Allocation' seems relatively robust to small changes in this parameter, as the objective function (Total Net Yield) changes by less than 5% for a 20% variation.")
else:
    print("Recommendation for 'Max Dealer Allocation' seems somewhat fragile to changes in this parameter, as the objective function (Total Net Yield) changes significantly for a 20% variation.")

# Confidence Statement
print("\n--- Confidence Statement ---")
print("The VP should be moderately confident in this recommendation. While the model is robust to changes in the minimum total allocation, it is somewhat fragile to the portfolio risk limit and the maximum allocation per dealer. The recommendation would change significantly if the acceptable risk level is lowered or if there are stricter limits on individual dealer allocations, forcing a different distribution of funds to maintain optimality. Therefore, the VP should closely scrutinize the reliability of the estimated risk scores and ensure the individual dealer limits are appropriate.")


--- Original Model Results ---
Total Net Yield: 1.4573
Total Allocated Volume: 25.00
Actual Average Risk Score: 2.63
Allocations:


,Allocated_Volume,Selected
Dealer_ID,,
Mercedes Dealer,3.0,1.0
Nissan Dealer,3.0,1.0
Toyota Dealer,3.0,1.0
Mazda Dealer,3.0,1.0
Dodge Dealer,1.0,1.0
Acura Dealer,3.0,1.0
Ford Dealer,3.0,1.0
Honda Dealer,3.0,1.0
Kia Dealer,3.0,1.0



--- Sensitivity Analysis: Minimum Total Allocation ---

Scenario 1: Minimum Total Allocation = $14.40M (-20%)
Total Net Yield: 1.4573
Total Allocated Volume: 25.00
Actual Average Risk Score: 2.63

Scenario 2: Minimum Total Allocation = $21.60M (+20%)
Total Net Yield: 1.4573
Total Allocated Volume: 25.00
Actual Average Risk Score: 2.63

Comparison for Minimum Total Allocation:
Original Net Yield: 1.4573
-20% Min Alloc Net Yield: 1.4573
+20% Min Alloc Net Yield: 1.4573
Recommendation for 'Minimum Total Allocation' seems relatively robust to small changes in this parameter, as the objective function (Total Net Yield) changes by less than 5% for a 20% variation.

--- Sensitivity Analysis: Portfolio Risk Limit ---

Scenario 3: Portfolio Risk Limit = 2.24 (-20%)
Total Net Yield: 1.0461
Total Allocated Volume: 20.26
Actual Average Risk Score: 2.24

Scenario 4: Portfolio Risk Limit = 3.36 (+20%)
Total Net Yield: 1.4573
Total Allocated Volume: 25.00
Actual Average Risk Score: 2.63

Comparison 

**Section 5** — Time Dimension

Show how your recommendation plays out over time. Approaches include:

*   Your model already includes time periods — describe the resulting schedule or sequence
*   Your recommendation is implemented in phases — show the rollout timeline
*   Demand or constraints vary across periods — show how the model handles this

If time genuinely is not a factor in your problem, explain why in one paragraph, and describe the circumstances under which it would become relevant.

The current optimization model is designed as a static, single-period allocation problem. It focuses on optimizing the lending budget for a single quarter (as implied by the constraints of \$25M per quarter" and "\$3M per quarter"). Therefore, time is not explicitly modeled as a dynamic factor influencing decision variables or parameters within the current formulation. The recommendations generated are based on a snapshot of dealer performance (yield and risk) and policy constraints for that specific period.

However, time would become highly relevant under several circumstances:

1.  **Changing Dealer Performance**: If the potential yield rates or risk scores of dealerships are expected to change significantly over subsequent quarters due to market shifts, economic cycles, or dealer-specific improvements/deteriorations, a dynamic, multi-period model would be necessary to capture these evolving characteristics and re-optimize allocations over time.
2.  **Sequential Decision-Making**: If the VP needs to make allocation decisions for multiple future quarters, taking into account how current allocations might influence future options (e.g., building relationships with certain dealers, or managing overall portfolio growth trajectories), a multi-period optimization framework would be required.
3.  **Budget Fluctuations**: If the total lending budget itself varies quarter-to-quarter, or if unallocated funds can be rolled over, a time dimension would allow the model to plan for these changes and smooth allocations.
4.  **Seasonality**: In many lending markets, demand and risk can exhibit seasonal patterns. Incorporating seasonality would enable the model to adjust allocations to capitalize on peak demand or mitigate risk during tougher periods.

**Section 6** — Where This Model Simplifies Reality

Identify at least one relationship in your problem where the linear assumption is suspect. Where would you expect diminishing returns? Where does doubling an input not double the output?

You do not need to solve a nonlinear model. You do need to show that you understand where your model's assumptions would mislead a decision-maker if taken at face value. This section should feel like an honest caveat, not a checkbox.